In [ ]:
import pandas as pd
import numpy as np

from folktables import ACSDataSource, ACSIncome, ACSTravelTime
from collections import Counter

In [ ]:
# years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
years = [2018]

state_train = "TX"
train_year = 2018

_yearly_data = {}


data_source = ACSDataSource(
    survey_year=str(train_year), horizon="1-Year", survey="person"
)
acs_data = data_source.get_data(states=[state_train], download=True)

_yearly_data[train_year] = ACSTravelTime.df_to_numpy(acs_data)

for y in years:
    try:
        data_source = ACSDataSource(
            survey_year=str(y), horizon="1-Year", survey="person"
        )
        acs_data = data_source.get_data(states=["TX"], download=True)

        _yearly_data[y] = ACSTravelTime.df_to_numpy(acs_data)
    except:
        pass

In [ ]:
X, y, z = _yearly_data[train_year]

In [ ]:
aif_df = pd.DataFrame()
for i in range(len(X[0])):
    aif_df["X_%i" % i] = X[:, i]

aif_df["race"] = z
aif_df["<20min"] = y
aif_df["state"] = state_train

In [ ]:
# aif_df.to_csv(f"./{state_train}.csv")

In [ ]:
aif_df.head()

In [ ]:
_yearly_data.keys()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

###### Your favorite learning algorithm here #####
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X, y)


for y2 in _yearly_data.keys():
    X_test, y_test, group_test = _yearly_data[y2]

    y_pred = model.predict(X_test)

    white_tpr = np.mean(y_pred[(y_test == 1) & (group_test == 1)])
    black_tpr = np.mean(y_pred[(y_test == 1) & (group_test == 2)])
    print(
        white_tpr,
        black_tpr,
    )
    print("%i - %f" % (y2, white_tpr - black_tpr))

In [ ]:
X.shape

In [ ]:
cols_X = ["X_%i" % i for i in range(len(X[0]))]

aif_df = pd.DataFrame()

In [ ]:
for i in range(len(X[0])):
    aif_df["X_%i" % i] = X[:, i]

aif_df["race"] = z
aif_df["<20min"] = y
aif_df["state"] = state_train

In [ ]:
aif_df

In [ ]:
aif_df.to_csv(f"./{state_train}.csv")

In [ ]:
aif_df.columns

In [ ]:
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from aif360.algorithms.preprocessing.reweighing import Reweighing

_label_names = ["<20min"]
_protected_attribute_names = ["race"]
_favorable_label = 1
_unfavorable_label = 0


original_data = BinaryLabelDataset(
    df=aif_df,
    label_names=_label_names,
    protected_attribute_names=_protected_attribute_names,
    favorable_label=_favorable_label,
    unfavorable_label=_unfavorable_label,
)

privileged_groups = [{"race": 1}]
unprivileged_groups = [{"race": 2}]

In [ ]:
RW = Reweighing(
    unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups
)
RW.fit(original_data)
transformed_data = RW.transform(original_data)

In [ ]:
w = transformed_data.instance_weights

###### Your favorite learning algorithm here #####
fair_model = make_pipeline(StandardScaler(), RandomForestClassifier())
fair_model.fit(X, y, randomforestclassifier__sample_weight=w)

In [ ]:
def test_on_state_data(state_name, years, _model):
    # Evaluate model on 2015-2018 data
    accuracies = []
    eq_opps = []
    tnrs = []

    for year in [2015, 2016, 2017, 2018]:
        data_source = ACSDataSource(survey_year=year, horizon="1-Year", survey="person")
        acs_data = data_source.get_data(states=[state_name], download=True)
        features, labels, group_test = ACSTravelTime.df_to_numpy(acs_data)

        y_pred = _model.predict(features)

        accuracies.append(_model.score(features, labels))

        white_tpr = np.mean(y_pred[(labels == 1) & (group_test == 1)])
        black_tpr = np.mean(y_pred[(labels == 1) & (group_test == 2)])

        white_fnr = 1 - np.mean(y_pred[(labels == 0) & (group_test == 1)])
        black_fnr = 1 - np.mean(y_pred[(labels == 0) & (group_test == 2)])

        eq_opps.append(white_tpr - black_tpr)
        tnrs.append(white_fnr - black_fnr)

    return accuracies, eq_opps, tnrs

In [ ]:
test_on_state_data("CA", [2015, 2016, 2017, 2018], model)

In [ ]:
test_on_state_data("CA", [2015, 2016, 2017, 2018], fair_model)

In [ ]:
test_on_state_data("NY", [2015, 2016, 2017, 2018], model)

In [ ]:
test_on_state_data("NY", [2015, 2016, 2017, 2018], fair_model)

In [ ]:
abbreviations = [
    # https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States#States.
    "AK",
    "AL",
    "AR",
    "AZ",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "IA",
    "ID",
    "IL",
    "IN",
    "KS",
    "KY",
    "LA",
    "MA",
    "MD",
    "ME",
    "MI",
    "MN",
    "MO",
    "MS",
    "MT",
    "NC",
    "ND",
    "NE",
    "NH",
    "NJ",
    "NM",
    "NV",
    "NY",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VA",
    "VT",
    "WA",
    "WI",
    "WV",
    "WY",
    # https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States#Federal_district.
    "DC",
    # https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States#Inhabited_territories.
    "AS",
    "GU",
    "MP",
    "PR",
    "VI",
]

In [ ]:
smaller_acc = 1
bigger_acc = 0
best_s = ""
worst_s = ""

for s in abbreviations:
    try:
        ret = test_on_state_data(s, [2015], model)

        if ret[0][0] > bigger_acc:
            bigger_acc = ret[0][0]
            best_s = s

        if ret[0][0] < smaller_acc:
            smaller_acc = ret[0][0]
            worst_s = s
    except:
        pass

In [ ]:
print(best_s, worst_s)

In [ ]:
import pandas as pd

from sklearn.cluster import DBSCAN, KMeans

filenameFinal = "Clusteringv3.xlsx"

finalDF = pd.read_excel(filenameFinal)


N_CLUSTERS = 5
db = KMeans(n_clusters=N_CLUSTERS, random_state=0, n_init="auto").fit(
    finalDF.to_numpy()
)

db = DBSCAN(eps=0.7, min_samples=1, metric="precomputed").fit(finalDF.to_numpy())
# db = KMeans(n_clusters=2, random_state=0, n_init="auto").fit(finalDF.to_numpy())

print(db.labels_)

In [ ]:
_X = finalDF.to_numpy()
for i in range(len(_X)):
    if (_X[i] < 0.5).any():
        print(i)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(finalDF.to_numpy().ravel())

In [ ]:
f = "acs_travel_time.csv"


state_train = "CA"
state_test = "NY"
train_year = 2014

_yearly_data = {}

_states = []
_states.append("CA")


data_source = ACSDataSource(
    survey_year=str(train_year), horizon="1-Year", survey="person"
)
acs_data = data_source.get_data(states=[state_train], download=True)

acs_data["_state"] = state_train
acs_data["_year"] = 2014

for y in years:
    try:
        data_source = ACSDataSource(
            survey_year=str(y), horizon="1-Year", survey="person"
        )
        acs_data2 = data_source.get_data(states=[state_test], download=True)
        acs_data2["_state"] = state_test
        acs_data2["_year"] = y

        acs_data = pd.concat((acs_data, acs_data2))

    except:
        pass

In [ ]:
acs_data

In [ ]:


_yearly_data[train_year] = 

for y in years:
    
    try:
        data_source = ACSDataSource(survey_year=str(y), horizon='1-Year', survey='person')
        acs_data = data_source.get_data(states=["NY"], download=True)

        _yearly_data[y] = ACSTravelTime.df_to_numpy(acs_data)
    except:
        pass    



In [ ]:
from folktables import ACSDataSource, ACSEmployment

data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
acs_data = data_source.get_data(states=["AL"], download=True)
features, label, group = ACSEmployment.df_to_numpy(acs_data)

In [ ]:
group

# Distribution shift across Time

In [ ]:
from folktables import ACSDataSource, ACSPublicCoverage
from sklearn.linear_model import LogisticRegression

# Download 2014 data
data_source = ACSDataSource(survey_year=2014, horizon="1-Year", survey="person")
acs_data14 = data_source.get_data(states=["TX"], download=True)
features14, labels14, _ = ACSPublicCoverage.df_to_numpy(acs_data14)

# Train model on 2014 data
# Plug-in your method for tabular datasets
model = LogisticRegression()
model.fit(features14, labels14)

# Evaluate model on 2015-2018 data
accuracies = []
for year in [2015, 2016, 2017, 2018]:
    data_source = ACSDataSource(survey_year=year, horizon="1-Year", survey="person")
    acs_data = data_source.get_data(states=["CA"], download=True)
    features, labels, group = ACSPublicCoverage.df_to_numpy(acs_data)
    accuracies.append(model.score(features, labels))

# Distribution shift across states

In [ ]:
from folktables import ACSDataSource, ACSIncome
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
ca_data = data_source.get_data(states=["CA"], download=True)
ca_features, ca_labels, _ = ACSIncome.df_to_numpy(ca_data)

X_train, X_test, y_train, y_test, group_train, group_test = train_test_split(
    ca_features, ca_labels, _, test_size=0.2, random_state=0
)

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(ca_features, ca_labels)

# Test on CA data
print(model.score(ca_features, ca_labels))

# compute TPR for CA
yhat = model.predict(X_test)
white_tpr = np.mean(yhat[(y_test == 1) & (group_test == 1)])
black_tpr = np.mean(yhat[(y_test == 1) & (group_test == 2)])

print(white_tpr - black_tpr)

# .------------------.

# Test on MI data


mi_data = data_source.get_data(states=["NY"], download=True)
mi_features, mi_labels, _ = ACSIncome.df_to_numpy(mi_data)
X_train_mi, X_test_mi, y_train_mi, y_test_mi, group_train_mi, group_test_mi = (
    train_test_split(mi_features, mi_labels, _, test_size=0.2, random_state=0)
)

print(model.score(mi_features, mi_labels))

# compute TPR for MI
yhat = model.predict(X_test_mi)
white_tpr = np.mean(yhat[(y_test_mi == 1) & (group_test_mi == 1)])
black_tpr = np.mean(yhat[(y_test_mi == 1) & (group_test_mi == 2)])

print(white_tpr - black_tpr)

# Create dataset

In [1]:
import pandas as pd
import numpy as np
import folktables
from folktables import ACSDataSource, ACSTravelTime
from collections import Counter

In [2]:
def adult_filter(data):
    """Mimic the filters in place for Adult data.

    Adult documentation notes: Extraction was done by Barry Becker from
    the 1994 Census database. A set of reasonably clean records was extracted
    using the following conditions:
    ((AAGE>16) && (AGI>100) && (AFNLWGT>1)&& (HRSWK>0))
    """
    df = data
    df = df[df["AGEP"] > 16]
    df = df[df["PINCP"] > 100]
    df = df[df["WKHP"] > 0]
    df = df[df["PWGTP"] >= 1]
    return df


ACSIncome = folktables.BasicProblem(
    features=[
        "AGEP",
        "COW",
        "SCHL",
        "MAR",
        "OCCP",
        "POBP",
        "RELP",
        "WKHP",
        "SEX",
        "RAC1P",
    ],
    target="PINCP",
    target_transform=lambda x: x > 50000,
    group="SEX",
    preprocess=adult_filter,
    postprocess=lambda x: np.nan_to_num(x, -1),
)

In [3]:
abbreviations = [
    # https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States#States.
    "AK",
    "AL",
    "AR",
    "AZ",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "IA",
    "ID",
    "IL",
    "IN",
    "KS",
    "KY",
    "LA",
    "MA",
    "MD",
    "ME",
    "MI",
    "MN",
    "MO",
    "MS",
    "MT",
    "NC",
    "ND",
    "NE",
    "NH",
    "NJ",
    "NM",
    "NV",
    "NY",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VA",
    "VT",
    "WA",
    "WI",
    "WV",
    "WY",
    "DC",
    "AS",
    "GU",
    "MP",
    "PR",
    "VI",
]

In [9]:
# years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
# years = [2018]
_yearly_data = {}
train_year = 2018
# state_train = "MI"
# for y in years:
#     try:
data_source = ACSDataSource(
    survey_year=str(train_year), horizon="1-Year", survey="person"
)
for state in abbreviations:
    try:
        acs_data = data_source.get_data(states=[state], download=True)

        _yearly_data[state] = ACSIncome.df_to_numpy(acs_data)

        X, y, z = _yearly_data[state]

        aif_df = pd.DataFrame()
        for i in range(len(X[0])):
            aif_df["X_%i" % i] = X[:, i]

        aif_df["Gender"] = [0 if i == 1 else 1 for i in z]
        aif_df[">50K"] = [1 if i == True else 0 for i in y]
        # aif_df["state"] = state_train

        aif_df.to_csv(f"./Income_acs/{state}.csv")
    except:
        continue

In [73]:
X, y, z = _yearly_data[train_year]

In [74]:
aif_df = pd.DataFrame()
for i in range(len(X[0])):
    aif_df["X_%i" % i] = X[:, i]

aif_df["Gender"] = [0 if i == 1 else 1 for i in z]
aif_df[">50K"] = [1 if i == True else 0 for i in y]
aif_df["state"] = state_train

In [75]:
aif_df[(aif_df["Gender"] == 1) & (aif_df[">50K"] == 1)].shape[0]

5850

In [76]:
aif_df[(aif_df["Gender"] == 1) & (aif_df[">50K"] == 0)].shape[0]

17902

In [77]:
aif_df.head()

,X_0,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,Gender,>50K,state
0,20.0,2.0,19.0,5.0,2752.0,26.0,17.0,6.0,2.0,1.0,1,0,MI
1,20.0,1.0,18.0,5.0,2006.0,26.0,17.0,40.0,1.0,1.0,0,0,MI
2,81.0,2.0,23.0,5.0,2205.0,46.0,17.0,50.0,1.0,1.0,0,0,MI
3,39.0,1.0,22.0,5.0,4230.0,303.0,17.0,45.0,2.0,1.0,1,1,MI
4,21.0,1.0,19.0,5.0,5860.0,6.0,17.0,14.0,1.0,9.0,0,0,MI


In [78]:
aif_df.shape

(50008, 13)

In [79]:
aif_df.to_csv(f"./{state_train}.csv")

In [ ]:
mbnkjn

# Travel Time

In [1]:
# Define the mapping of occupation codes to broader categories
occupation_category_mapping = {
    "Management, Business, Science, and Arts Occupations": range(0, 3541),
    "Service Occupations": range(3541, 4651),
    "Sales and Office Occupations": range(4651, 5941),
    "Natural Resources, Construction, and Maintenance Occupations": range(5941, 7641),
    "Production, Transportation, and Material Moving Occupations": range(7641, 9750),
    "Military Specific Occupations": range(9750, 50000),
}


# Define the function to filter and convert the list
def categorize_occupations(occupation_codes):
    category_to_number = {
        "Management, Business, Science, and Arts Occupations": 1,
        "Service Occupations": 2,
        "Sales and Office Occupations": 3,
        "Natural Resources, Construction, and Maintenance Occupations": 4,
        "Production, Transportation, and Material Moving Occupations": 5,
        "Military Specific Occupations": 6,
    }

    categorized_codes = []

    for code in occupation_codes:
        category_number = None
        for category, code_range in occupation_category_mapping.items():
            if int(code) in code_range:
                category_number = category_to_number[category]
                break
        categorized_codes.append(category_number)
        if category_number is None:
            raise ValueError(f"Occupation code {code} not found in any category")

    return categorized_codes

In [2]:
def compute_disparity(
    sensitive_attributes, targets, possible_sensitive_attributes, possible_targets
):
    disparities = []
    # Compute the disparity
    for target in possible_targets:
        for sensitive_attribute in possible_sensitive_attributes:
            Z_equal_z = len(
                sensitive_attributes[sensitive_attributes == sensitive_attribute]
            )
            Z_not_equal_z = len(sensitive_attributes) - Z_equal_z

            Z_equal_z_and_Y_equal_target = len(
                sensitive_attributes[
                    (sensitive_attributes == sensitive_attribute) & (targets == target)
                ]
            )
            Z_not_equal_z_and_Y_equal_target = len(
                sensitive_attributes[
                    (sensitive_attributes != sensitive_attribute) & (targets == target)
                ]
            )
            disparities.append(
                abs(
                    Z_equal_z_and_Y_equal_target / Z_equal_z
                    - Z_not_equal_z_and_Y_equal_target / Z_not_equal_z
                )
            )
    print(max(disparities))
    return disparities

In [3]:
import pandas as pd
import numpy as np
import folktables
from folktables import ACSDataSource, ACSTravelTime
from collections import Counter

In [4]:
def travel_time_filter(data):
    """
    Filters for the employment prediction task
    """
    df = data
    df = df[df["AGEP"] > 16]
    df = df[df["PWGTP"] >= 1]
    df = df[df["ESR"] == 1]
    return df


ACSTravelTime = folktables.BasicProblem(
    features=[
        "AGEP",
        "SCHL",
        "MAR",
        "SEX",
        "DIS",
        "ESP",
        "MIG",
        "RELP",
        "RAC1P",
        "PUMA",
        "ST",
        "CIT",
        "OCCP",
        "JWTR",
        "POWPUMA",
        "POVPIP",
    ],
    target="JWMNP",
    target_transform=lambda x: x > 20,
    group="SEX",
    preprocess=travel_time_filter,
    postprocess=lambda x: np.nan_to_num(x, -1),
)

In [7]:
# years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
years = [2018]
_yearly_data = {}
train_year = 2018

state_list = [
    "AK",
    "AL",
    "AR",
    "AZ",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "IA",
    "ID",
    "IL",
    "IN",
    "KS",
    "KY",
    "LA",
    "MA",
    "MD",
    "ME",
    "MI",
    "MN",
    "MO",
    "MS",
    "MT",
    "NC",
    "ND",
    "NE",
    "NH",
    "NJ",
    "NM",
    "NV",
    "NY",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VA",
    "VT",
    "WA",
    "WI",
    "WV",
    "WY",
    "DC",
    "AS",
    "GU",
    "MP",
    "PR",
    "VI",
]
features = [
    "AGEP",
    "SCHL",
    "MAR",
    "SEX",
    "DIS",
    "ESP",
    "MIG",
    "RELP",
    "RAC1P",
    "PUMA",
    "ST",
    "CIT",
    "OCCP",
    "JWTR",
    "POWPUMA",
    "POVPIP",
]

for state in state_list:
    for y in years:
        try:
            data_source = ACSDataSource(
                survey_year=str(y), horizon="1-Year", survey="person"
            )
            acs_data = data_source.get_data(states=[state], download=True)

            _yearly_data[y] = ACSTravelTime.df_to_numpy(acs_data)
        except:
            pass

        X, y, z = _yearly_data[train_year]
        aif_df = pd.DataFrame()

        for i in range(len(X[0])):
            aif_df["X_%i" % i] = X[:, i]

        aif_df["Gender"] = [0 if i == 1 else 1 for i in z]
        aif_df[">20min"] = [1 if i == True else 0 for i in y]
        aif_df["state"] = state
        aif_df.columns = features + ["Gender", ">20min", "state"]
        aif_df["OCCP"] = categorize_occupations(list(aif_df["OCCP"]))
        print(
            state,
            " ",
            compute_disparity(
                np.array(aif_df["Gender"]), np.array(aif_df[">20min"]), [0, 1], [0, 1]
            ),
        )
        aif_df.to_csv(f"./travel_acs/{state}.csv")

0.061501807126733726
AK   [0.0615018071267337, 0.0615018071267337, 0.061501807126733726, 0.061501807126733726]
0.07759126932661614
AL   [0.07759126932661609, 0.07759126932661609, 0.07759126932661614, 0.07759126932661614]
0.05660468201431712
AR   [0.05660468201431712, 0.05660468201431712, 0.056604682014317065, 0.056604682014317065]
0.06820369229670159
AZ   [0.06820369229670153, 0.06820369229670153, 0.06820369229670159, 0.06820369229670159]
0.06822338611016954
CA   [0.06822338611016954, 0.06822338611016954, 0.06822338611016948, 0.06822338611016948]
0.0638799885848742
CO   [0.0638799885848742, 0.0638799885848742, 0.0638799885848742, 0.0638799885848742]
0.07016615475829979
CT   [0.07016615475829979, 0.07016615475829979, 0.07016615475829979, 0.07016615475829979]
0.05923724307998601
DE   [0.05923724307998601, 0.05923724307998601, 0.05923724307998601, 0.05923724307998601]
0.05069014128290117
FL   [0.05069014128290117, 0.05069014128290117, 0.05069014128290111, 0.05069014128290111]
0.0634459421

In [ ]:
    aif_df.head()